# Explore: predict *play* instead of *own*

**Question:** if positives are games a user actually played (≥ X minutes), how do simple
baselines behave, and does that change our approach?

Everything is built **in this notebook** from the prepped cache, reusing `split_by_user`
and `ranking_candidates`; nothing in `src/` changes until we decide. For each threshold:

- **positives:** held-out *played* games (leave-one-out among played games)
- **negatives:** 99 popularity-weighted games the user owns in **no** split
- **owned but not played:** stays in train as context; never a positive or a negative

Threshold `0` is today's task (any owned game is a positive), for comparison.

In [28]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from assignment_2.modules.config import get_config
from assignment_2.modules.core import ranking_per_user
from assignment_2.modules.data import load_prepped, ranking_candidates
from assignment_2.modules.data.split import split_by_user

get_config.cache_clear()
cfg = get_config()
t = load_prepped(cfg)
owned = pd.concat([t[f"interactions_{s}"] for s in ("train", "valid", "test")], ignore_index=True)
print(f"sample={cfg.sample.tag}  negatives={cfg.evaluation.negatives}  "
      f"{len(owned):,} ownership rows, {owned.user_id.nunique():,} users")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
sample=users_0.1  negatives=popularity  490,846 ownership rows, 7,058 users


## 1. Build a "predict play" task for a threshold

Validation only: we're exploring, so test stays untouched.

In [29]:
THRESHOLDS = [0, 5, 120]   # minutes; 0 = any owned game counts (today's task)
K = 10


def play_task(min_minutes: int) -> dict[str, pd.DataFrame]:
    """Train rows, held-out played games, and validation candidates for one threshold."""
    if min_minutes == 0:
        played_mask = pd.Series(True, index=owned.index)   # every owned game is a positive
    else:
        played_mask = owned.playtime_forever >= min_minutes
    played, unplayed = owned[played_mask], owned[~played_mask]

    s = split_by_user(played, cfg.split, cfg.seed)          # leave-one-out among played games
    train = pd.concat([s.train, unplayed], ignore_index=True)  # unplayed: context, never held out
    cand = ranking_candidates(
        held_out=s.valid,
        train=train,
        owned=owned,                                        # never sample anything the user owns
        n_negatives=cfg.evaluation.n_negatives,
        seed=cfg.seed + 1,
        negatives=cfg.evaluation.negatives,
    )
    return {"train": train, "valid": s.valid, "cand": cand, "min_minutes": min_minutes}


tasks = {m: play_task(m) for m in THRESHOLDS}
pd.DataFrame(
    {
        m: {
            "train rows": len(d["train"]),
            "users evaluated": d["cand"].user_id.nunique(),
            "candidate rows": len(d["cand"]),
        }
        for m, d in tasks.items()
    }
).T.rename_axis("min minutes")

,train rows,users evaluated,candidate rows
min minutes,,,
0,477762,6534,653400
5,478354,6233,623300
120,478906,5949,594900


**Check:** fewer users are evaluated at higher thresholds (a user needs ≥ 3 *played* games
to have one held out). How many do we lose at 120 minutes?

## 2. Baselines

Each baseline is a function `scores(task) -> np.ndarray`, one score per candidate row, in
order. `evaluate` turns scores into Hit@10 / NDCG@10 / MRR averaged over users.

In [30]:
def evaluate(task: dict, scores: np.ndarray) -> pd.Series:
    c = task["cand"]
    per_user = ranking_per_user(c.user_id, c.label, scores, k=K, seed=cfg.seed)
    return pd.Series({m: v.mean() for m, v in per_user.items()})


def random_scores(task: dict) -> np.ndarray:
    return np.random.default_rng(cfg.seed).random(len(task["cand"]))


def owner_popularity(task: dict) -> np.ndarray:
    """How many train users own each candidate (played or not)."""
    counts = task["train"].item_id.value_counts()
    return task["cand"].item_id.map(counts).fillna(0).to_numpy(float)

### 2a. Play popularity
**Idea:** rank by how many train users *played* each game (≥ the threshold), not just
own it. If positives are played games, does counting plays beat counting owners?

**Hint:** same as `owner_popularity`, but filter `task["train"]` to rows with
`playtime_forever >= min_minutes` first. You'll need the threshold: pass it in, or store it
in the task dict.

In [31]:
def play_popularity(task: dict) -> np.ndarray:
    """How many train users *played* each candidate (at this task's threshold)."""
    train = task["train"]
    played = train[train.playtime_forever >= task["min_minutes"]]
    counts = played.item_id.value_counts()
    return task["cand"].item_id.map(counts).fillna(0).to_numpy(float)

### 2b. Play rate
**Idea:** a game owned by 1,000 people but played by 100 is less promising than one owned
by 200 and played by 180. Score = share of a game's owners who played it.

**Watch out:** a game with 1 owner who played it has a rate of 100%. Shrink toward the
overall rate: `(plays + a * overall_rate) / (owners + a)` with `a` around 10. Which games
does the shrinkage protect against?

In [32]:
def play_rate(task: dict, a: float = 10.0) -> np.ndarray:
    """Share of a game's train owners who played it, shrunk toward the overall rate."""
    train = task["train"]
    owners = train.item_id.value_counts()                     # everyone who owns it
    plays = (train[train.playtime_forever >= task["min_minutes"]]
             .item_id.value_counts()
             .reindex(owners.index, fill_value=0))            # same games, 0 if never played
    overall = plays.sum() / owners.sum()                      # the rate across all ownership

    rate = (plays + a * overall) / (owners + a)               # shrunk toward `overall`
    return task["cand"].item_id.map(rate).fillna(overall).to_numpy(float)


### 2c. Personal: similarity to what the user played
**Idea:** score a candidate by its max (or mean) Jaccard similarity to the games *this user*
played in train, where a game is represented by the set of users who played it.
This is the first **personalized** baseline: everything above ranks the same for every user.

**Hints:**
- build `users_of = played_train.groupby("item_id").user_id.agg(set)` and
  `played_by = played_train.groupby("user_id").item_id.agg(set)`
- Jaccard(a, b) = |A ∩ B| / |A ∪ B|
- it's slow over 650K candidate rows × a library each; start with a sample of users, or
  limit each user's history to their most-played ~20 games

In [33]:
def jaccard(a: set, b: set) -> float:
    """|A ∩ B| / |A ∪ B|; 0 for two empty sets."""
    inter = len(a & b)
    union = len(a) + len(b) - inter
    return inter / union if union else 0.0

In [34]:
def jaccard_to_history(task: dict) -> np.ndarray:
    train = task["train"]
    played_train = train[train.playtime_forever >= task["min_minutes"]]
    users_of = played_train.groupby("item_id").user_id.agg(set).to_dict()
    played_by = played_train.groupby("user_id").item_id.agg(set).to_dict()

    scores = np.zeros(len(task["cand"]))
    for row, (u, i) in enumerate(zip(task["cand"].user_id, task["cand"].item_id)):
        players_i = users_of.get(i)
        if not players_i:                 # nobody played this candidate: score stays 0
            continue
        scores[row] = max(
            (jaccard(players_i, users_of[j]) for j in played_by.get(u, ())),
            default=0.0,
        )
    return scores

## 3. Compare across thresholds

In [35]:
BASELINES = {
    "random": random_scores,
    "owner popularity": owner_popularity,
    "play popularity": play_popularity,
    "play rate": play_rate,
    "jaccard to history": jaccard_to_history,
}

results = pd.concat(
    {
        (m, name): evaluate(task, fn(task))
        for m, task in tasks.items()
        for name, fn in BASELINES.items()
    },
    names=["min_minutes", "baseline"],
).unstack().round(3)
results

hit@10  ndcg@10    mrr
min_minutes baseline                                  
0           jaccard to history   0.494    0.360  0.337
            owner popularity     0.289    0.170  0.155
            play popularity      0.289    0.170  0.155
            play rate            0.100    0.046  0.052
            random               0.091    0.041  0.049
5           jaccard to history   0.432    0.261  0.231
            owner popularity     0.306    0.172  0.153
            play popularity      0.338    0.195  0.175
            play rate            0.255    0.132  0.120
            random               0.090    0.041  0.049
120         jaccard to history   0.490    0.298  0.261
            owner popularity     0.351    0.208  0.187
            play popularity      0.399    0.239  0.214
            play rate            0.345    0.186  0.162
            random               0.090    0.041  0.049

## 4. Questions to answer before deciding
- Does **owner popularity** get worse as the threshold rises? (If played games are a
  different set from popular-to-own games, it should.)
- Does **play popularity** or **play rate** beat owner popularity at 5 and 120 minutes? If yes,
  playtime carries signal the ownership task couldn't use.
- How much does **Jaccard** (personal) add over the best popularity-style score, and at which
  threshold is that gap largest? That's where personalized models have the most room.
- How many users does 120 minutes cost? Is the remaining sample still big enough?

**Takeaways:**
-